# Phase 1 Statistical Results — Discussion
**Drought Index Selection: ERA5 Historical Analysis**

GWSC Drought Project

In [ ]:
# Cell 1: Load packages
library(httr2)
library(dplyr)
library(tidyr)
library(ggplot2)
library(knitr)
cat('Packages loaded\n')

: 

In [ ]:
# Cell 2: Connect to Databricks Volume and fetch all RDS outputs
DB_HOST     <- "https://dbc-9c907c32-f434.cloud.databricks.com"
DB_TOKEN    <- Sys.getenv("DATABRICKS_TOKEN")
VOLUME_PATH <- "/Volumes/projects/gwsc_drought/drought_outputs"

read_db_rds <- function(filename) {
  url <- paste0(DB_HOST, "/api/2.0/fs/files", VOLUME_PATH, "/", filename)
  tmp <- tempfile(fileext = ".rds")
  tryCatch({
    request(url) |>
      req_auth_bearer_token(DB_TOKEN) |>
      req_options(http_version = 2L) |>
      req_perform(path = tmp)
    readRDS(tmp)
  }, error = function(e) {
    message("Could not load ", filename, ": ", conditionMessage(e))
    NULL
  })
}

cat("Fetching RDS files from Volume...\n")
friedman_strat <- read_db_rds("analysis1_friedman_stratified.rds")
avg_ranks      <- read_db_rds("analysis1_avg_ranks.rds")
nemenyi_pairs  <- read_db_rds("analysis1_nemenyi_pairs.rds")
equiv_groups   <- read_db_rds("analysis1_equivalence_groups.rds")
thr_robustness <- read_db_rds("analysis1_threshold_robustness.rds")
pct_change     <- read_db_rds("analysis2_pct_change.rds")
rank_stability <- read_db_rds("analysis2_rank_stability.rds")
cdd_spearman   <- read_db_rds("analysis3_cdd_spearman.rds")
cdd_summary    <- read_db_rds("analysis3_cdd_summary.rds")
annual_ref     <- read_db_rds("drought_annual_summary_ref.rds")
cat("Done.\n")

In [ ]:
# Cell 3: Column name diagnostic — paste output here so we can fix any mismatches
for (nm in c("friedman_strat","avg_ranks","nemenyi_pairs","equiv_groups",
             "thr_robustness","pct_change","rank_stability",
             "cdd_spearman","cdd_summary","annual_ref")) {
  obj <- get(nm)
  if (!is.null(obj)) {
    cat("\n===", nm, "===", "\n")
    cat("Columns:", paste(names(obj), collapse=", "), "\n")
    cat("Rows:", nrow(obj), "\n")
  } else {
    cat("\n===", nm, "=== NULL (failed to load)\n")
  }
}

---
# Analysis 1: Friedman Test — Do drought definitions agree?

**What the test does:** 15 configs (5 indices × 3 accumulation periods) run at each of 3 WMO thresholds (−0.5, −1.0, −1.5). Within each threshold, Friedman tests whether all 15 configs rank the same.

- **Friedman Q**: large = strong rank disagreement among configs
- **p-value**: <0.05 = significant differences exist
- **Kendall W**: 0 = total disagreement, 1 = perfect rank agreement

In [ ]:
# Cell 4: Friedman results — raw print
print(friedman_strat)

In [ ]:
# Cell 5: Friedman summary — Kendall W by characteristic
df <- as.data.frame(friedman_strat)
if (!"threshold" %in% names(df)) {
  # 15 rows = 3 thresholds x 5 characteristics
  df$threshold <- rep(c(-0.5, -1.0, -1.5), each = 5)
}

cat("All p < 0.001:", all(df$p_value < 0.001, na.rm=TRUE), "\n\n")

cat("Kendall W by characteristic (across thresholds):\n")
df %>%
  group_by(characteristic) %>%
  summarise(W_min  = round(min(kendall_W), 3),
            W_mean = round(mean(kendall_W), 3),
            W_max  = round(max(kendall_W), 3)) %>%
  arrange(desc(W_mean)) %>%
  print()

cat("\nKendall W by threshold:\n")
df %>%
  group_by(threshold) %>%
  summarise(W_min  = round(min(kendall_W), 3),
            W_mean = round(mean(kendall_W), 3),
            W_max  = round(max(kendall_W), 3)) %>%
  print()

In [ ]:
# Cell 6: Friedman heatmap — Kendall W by threshold x characteristic
df <- as.data.frame(friedman_strat)
if (!"threshold" %in% names(df)) {
  df$threshold <- rep(c(-0.5, -1.0, -1.5), each = 5)
}

ggplot(df, aes(x = factor(threshold), y = characteristic, fill = kendall_W)) +
  geom_tile(colour = "white", linewidth = 0.5) +
  geom_text(aes(label = sprintf("W=%.3f", kendall_W)), size = 3.5, fontface = "bold") +
  scale_fill_gradient2(low = "#d73027", mid = "#fee08b", high = "#1a9850",
                       midpoint = 0.5, limits = c(0, 1), name = "Kendall W") +
  labs(title = "Friedman Test: Kendall W heatmap",
       subtitle = "Green = strong rank agreement across configs | Red = high disagreement",
       x = "WMO threshold", y = "Drought characteristic") +
  theme_minimal(base_size = 12)

---
# Analysis 1 continued: Average ranks and equivalence groups (Nemenyi)

In [ ]:
# Cell 7: Average ranks per config
print(avg_ranks)

In [ ]:
# Cell 8: Equivalence groups (configs not significantly different from each other)
print(equiv_groups)

In [ ]:
# Cell 9: Threshold robustness (Spearman rank correlation across thresholds)
print(thr_robustness)

---
# Analysis 2: Has drought changed? (Cross-period comparison)

In [ ]:
# Cell 10: Percentage change table (ref period vs comparison period)
print(pct_change)

In [ ]:
# Cell 11: Rank stability across periods
print(rank_stability)

---
# Analysis 3: CDD validation

Do SPI/SPEI-based drought frequencies agree with Consecutive Dry Days (CDD) from ERA5?

In [ ]:
# Cell 12: CDD Spearman correlation (SPI/SPEI freq vs CDD freq per country)
print(cdd_spearman)

In [ ]:
# Cell 13: CDD summary stats
print(cdd_summary)

---
# Annual reference summary

In [ ]:
# Cell 14: Annual reference summary — one row per config per country
print(head(annual_ref, 30))

In [ ]:
# Cell 15: Write diagnostics.txt listing all column names
sink("diagnostics.txt")
for (nm in c("friedman_strat","avg_ranks","nemenyi_pairs","equiv_groups",
             "thr_robustness","pct_change","rank_stability",
             "cdd_spearman","cdd_summary","annual_ref")) {
  obj <- get(nm)
  cat("\n===", nm, "===\n")
  if (!is.null(obj)) {
    cat("Columns:", paste(names(obj), collapse=", "), "\n")
    print(obj)
  } else {
    cat("NULL\n")
  }
}
sink()
cat("diagnostics.txt written to:", getwd(), "\n")